In [155]:
import pandas as pd
import numpy as np
import re
from ast import literal_eval
from collections import defaultdict
from collections import Counter
import time
import os
from scipy.stats import expon
from scipy.stats import gamma
from scipy.stats import chisquare
from statsmodels.stats.multitest import fdrcorrection

In [90]:
# Graph class
models = {
	'gamma_random_breakage': [0.3987543483729932, # a
			1.999999999999, # loc
			1749495.5696758535 # scale
			]
}

class Graph:

	def __init__(self, dataset=None, col_format=None, amp_type="ecDNA", loc_type="feature"):
		"""
		Parameters:
			self (Graph) : Graph object 
			dataset (tsv file) : AA aggregated results file
			amp_type (str) : type of focal amplification (ecDNA, BFB, etc.)
			loc_type (str) : cell line or feature
		Return:
			None
		"""
		if dataset is None:
			print("Error: provide Amplicon Architect annotations")	
		else:
			self.amp_type = amp_type
			self.loc_type = loc_type
			self.nodes = []
			self.edges = []
			self.locs = self.ImportLocs('/Users/michael/Documents/GitHub/AmpliconRepository/caper/bed_files/hg38_genes.bed')

			self.CreateNodes(dataset, col_format)
			self.CreateEdges()

	def CreateNodes(self, dataset, col_format):
		"""
		# OUTDATED DESC
		Create a nodes_df by iterating through the dataset, adding new genes to 
		a list of dictionaries. Update the amplicons for existing genes, then 
		merge the list into the dataframe after processing.
				
		Parameters: 
			self (Graph) : Graph object 
			dataset (tsv file) : AA aggregated results file
		Return: 
			None
		"""
		
		ALL_GENES = 'All genes' if col_format == 'space' else 'All_genes'
		FEATURE_ID = 'Feature ID' if col_format == 'space' else 'Feature_ID'

		# dictionary for fast lookups of gene labels
		gene_index = {}

		# get subset of ecDNA features
		if self.loc_type == "feature":
			all_features = dataset[dataset['Classification'] == self.amp_type]
		# for each feature
		for __, row in all_features.iterrows():
			# get the genes in this feature
			oncogenes = set(self.ExtractGenes(str(row['Oncogenes'])))
			all_genes = self.ExtractGenes(str(row[ALL_GENES]))
			for gene in all_genes:
				feature = row[FEATURE_ID]
				sample = feature.split("_amplicon")[0]
				# if the gene has been seen, add feature and cell line to info
				if gene in gene_index:
					index = gene_index[gene]
					self.nodes[index]['features'].append(feature)
					self.nodes[index]['samples'].append(sample)
				# otherwise add the gene to self.nodes as a new row
				else:
					node_info = {
						'label': gene,
						'oncogene': str(gene in oncogenes),
						'features': [feature],
						'samples': [sample],
						# 'chromosome': self.locs[gene][0],
						# 'start_coord': self.locs[gene][1],
						# 'end_coord': self.locs[gene][2]
                	}
					self.nodes.append(node_info)
					gene_index[gene] = len(self.nodes) - 1
    	
		# remove potential duplicate features or cell lines
		for node_info in self.nodes:
			node_info['features'] = list(set(node_info['features']))
			node_info['samples'] = list(set(node_info['samples']))
	
		# concatenate all nodes as rows in df
		self.nodes_df = pd.DataFrame(self.nodes)
			
	def CreateEdges(self):
		"""
		# OUTDATED DESC
		Create edges by iterating through pairs of nodes in nodes_df based on 
		shared features. Calculate edge properties for each pair, and append 
		the results to the edges_df after processing all pairs.
				
		Parameters: 
			self (Graph) : Graph object 
		Return: 
			None
		"""
		# extract relevant columns
		labels = self.nodes_df['label'].values
		features = self.nodes_df['features'].values
		
		# build reverse index to map features to nodes
		feature_to_nodes = defaultdict(list)
		for index, amps in enumerate(features):
			for amp in amps:
				feature_to_nodes[amp].append(index)

		# collect potential node pairs, avoiding duplicates
		potential_pairs = set()
		for nodelist in feature_to_nodes.values():
			for i, node1 in enumerate(nodelist):
				for node2 in nodelist[i + 1:]:
					potential_pairs.add((node1, node2)) 

		# unzip pairs into source and target nodes
		pairs = list(potential_pairs)
		src_indices, tgt_indices = zip(*pairs)
		
		# retrieve the matching features for all source and target nodes
		src_features = [features[i] for i in src_indices]
		tgt_features = [features[j] for j in tgt_indices]
		
		# remove id information from feature labels
		for feature_list in src_features:
			feature_list = [f.split("_amplicon")[0] for f in feature_list]
		for feature_list in tgt_features:
			feature_list = [f.split("_amplicon")[0] for f in feature_list]

		# calculate all intersections and unions
		inters = [list(set(s) & set(t)) for s,t in zip(src_features, tgt_features)]
		unions = [list(set(s) | set(t)) for s,t in zip(src_features, tgt_features)]

		# filter pairs with non-empty intersections
		non_empty_mask = [len(inter) > 0 for inter in inters]
		src_filtered = [src_indices[i] for i, mask in enumerate(non_empty_mask) if mask]
		tgt_filtered = [tgt_indices[i] for i, mask in enumerate(non_empty_mask) if mask]
		inters_filtered = [inters[i] for i, mask in enumerate(non_empty_mask) if mask]
		unions_filtered = [unions[i] for i, mask in enumerate(non_empty_mask) if mask]

		# calculate weights
		weights = [len(i) / len(u) for i,u in zip(inters_filtered, unions_filtered)]

		# compute p and q values
		p_values = [self.PVal(labels[a], labels[b]) for a,b in zip(src_filtered, tgt_filtered)]
		q_values = self.QVal(p_values)

		# create edges dict and df
		self.edges = [
			{
				'source': labels[i],
				'target': labels[j],
				'weight': weights[idx],
				'inter': inters_filtered[idx],
				'union': unions_filtered[idx],
				'pval': p_values[idx],
				'qval': q_values[idx]
			}
			for idx, (i, j) in enumerate(zip(src_filtered, tgt_filtered))
		]
		self.edges_df = pd.DataFrame(self.edges)

	# helper functions
	# ----------------
	def ExtractGenes(self, input):
		"""
		Parameters: 
			input (str) : ['A', 'B', 'C'] or ["'A'", "'B'", "'C'"]
		Return: 
			list: ["A", "B", "C"]
		"""
		pattern = r"['\"]?([\w./-]+)['\"]?"
		genelist = re.findall(pattern, input)
		return genelist
	
	def ImportLocs(self, bed_file):
		"""
		return a dict of format: {gene (string): (start chromosome (string), 
		start coordinate (int))}
		"""
		gene_coords = pd.read_csv(bed_file, sep="\t", header=None)
		gene_coords['chr_num'] = gene_coords.apply(lambda row: row[0][-1], axis=1)
		locs = {}
		for i,row in gene_coords.iterrows():
			locs[row[3]] = (row['chr_num'], int(row[1]), int(row[2]))
		return locs
	
	def Distance(self, a, b):
		# not in gene dict
		if not a in self.locs or not b in self.locs:
			return 'N/A'
		# multi chr
		if self.locs[a][0] != self.locs[b][0]:
			return 'N/A'
		# single chr
		return abs(self.locs[a][1] - self.locs[b][1])

	def PVal(self, gene_a, gene_b, model='gamma_random_breakage'):
		"""
		generate p_val based on naive random breakage model, considering distance
		but not number of incidents of co-amplification
		"""
		d = self.Distance(gene_a, gene_b)
		if d == 'N/A': return d
		params = models[model]
		cdf = gamma.cdf(d, a=params[0], loc=params[1], scale=params[2])
		return 1-cdf
		# if model == 'expon':
		# 	d = distance(a, b)
		# 	if not type(d) is str:
		# 		cdf = expon.cdf(d, loc=params[0], scale=params[1])
		# 		return 1-cdf
	
	def QVal(self, p_values, alpha=0.05):
		# extract valid p-values
		valid_mask = [p != 'N/A' for p in p_values]
		valid_p_values = [p for p in p_values if p != 'N/A']

		# apply FDR correction only to valid p-values
		_, valid_q_values = fdrcorrection(valid_p_values, alpha=alpha)

		# reconstruct q_values with 'N/A' in the appropriate positions
		valid_q_iter = iter(valid_q_values)
		q_values = [next(valid_q_iter) if valid else -1 for valid in valid_mask]

		return q_values
		# q_values = ['N/A'] * len(p_values)
		# j = 0
		# for i, valid in enumerate(valid_mask):
		# 	if valid:
		# 		q_values[i] = valid_q_values[j]
		# 		j += 1
        
	def Nodes(self):
		try:
			return self.nodes
		except:
			print('Error: build graph')

	def Edges(self):
		try:
			return self.edges
		except:
			print('Error: build graph')

In [91]:
# construct graph
start = time.process_time()

amp_repo_datasets_dir = "/Users/michael/Downloads/amplicon_repo_datasets/"
ccle = pd.read_csv(amp_repo_datasets_dir + "ccle_aggregated_results.csv")
graph = Graph(ccle, col_format='space')

end = time.process_time()
print('Construct Graph:', end-start, 's')

# graph.Nodes_df().to_csv('test_nodes.csv', index=False)  
#     graph.Edges_df().to_csv('/Users/michael/Downloads/test_edges_022725.csv', index=False)  

#     print('Test:', graph.Distance('SEPT14', 'ZNF713')) # fails

Construct Graph: 6.959731999999999 s


In [92]:
total_samples = len(ccle[ccle['Classification']=='ecDNA'])
nodes = graph.Nodes()
edges = graph.Edges()

In [93]:
node_dict = {node['label']: node for node in nodes}

In [94]:
# observed counts

# Number of samples with…
# O1,1 = ecDNA co-amplifications of A and B = len(edges[A-B]['inter'])
# O1,2 = ecDNA amplification of A, but not B = len(nodes[A]['samples']) - len(edges[A-B]['inter'])
# O2,1 = ecDNA amplification of B, but not A = len(nodes[B]['samples']) - len(edges[A-B]['inter'])
# O2,2 = ecDNA amplification of neither A or B = total_samples - len(edges[A-B]['inter'])

In [127]:
def chi_squared(edge):
    pdD = graph.PVal(edge['source'], edge['target'])
    if pdD == 'N/A': return 'N/A', -1
        
    geneA_samples = len(node_dict[edge['source']]['samples'])
    geneB_samples = len(node_dict[edge['target']]['samples'])
    geneAB_samples = len(edge['inter'])
    fA = geneA_samples/total_samples
    fB = geneB_samples/total_samples

    # observed
    O11 = geneAB_samples
    O12 = geneA_samples - geneAB_samples
    O21 = geneB_samples - geneAB_samples
    O22 = total_samples - geneAB_samples
    obs = [O11, O12, O21, O22]
    
    # expected
    E11 = total_samples * fA * fB * pdD
    E12 = total_samples * fA * (1 - fB * pdD)
    E21 = total_samples * fB * (1 - fA * pdD)
    E22 = total_samples * (1 - fA) * (1 - fB)
    exp = [E11, E12, E21, E22] 
    exp_rescaled = [e * sum(obs)/sum(exp) for e in exp]

    p_val = chisquare(obs, f_exp=exp_rescaled, ddof=2)[1]
    d = graph.Distance(edge['source'], edge['target'])
    
    return d, p_val

In [68]:
node_dict[edges[i]['target']]

{'label': 'SNORA21',
 'oncogene': 'False',
 'features': ['UACC812_BREAST_amplicon1_ecDNA_3',
  'BT474_BREAST_amplicon1_ecDNA_3',
  'EFM192A_BREAST_amplicon1_ecDNA_2'],
 'samples': ['BT474_BREAST', 'EFM192A_BREAST', 'UACC812_BREAST']}

In [126]:
def chi_squared_dep(edge):
    pdD = graph.PVal(edge['source'], edge['target'])
    if pdD == 'N/A': return 'N/A', -1
        
    geneA_samples = len(node_dict[edge['source']]['samples'])
    geneB_samples = len(node_dict[edge['target']]['samples'])
    geneAB_samples = len(edge['inter'])
    fAandB = geneAB_samples/total_samples
    fA = geneA_samples/total_samples
    fB = geneB_samples/total_samples
    fAorB = fA + fB - fAandB

    # observed
    O11 = geneAB_samples
    O12 = geneA_samples - geneAB_samples
    O21 = geneB_samples - geneAB_samples
    O22 = total_samples - geneAB_samples
    obs = [O11, O12, O21, O22]
    
    # expected
    E11 = total_samples * fAorB * pdD
    E12 = total_samples * fA * (1 - fB * pdD)
    E21 = total_samples * fB * (1 - fA * pdD)
    E22 = total_samples * (1 - fAorB)
    exp = [E11, E12, E21, E22] 
    exp_rescaled = [e * sum(obs)/sum(exp) for e in exp]

    p_val = chisquare(obs, f_exp=exp_rescaled, ddof=2)[1]
    d = graph.Distance(edge['source'], edge['target'])
    
    return d, p_val

In [96]:
edges[0]

{'source': 'BORCS8-MEF2B',
 'target': 'HAPLN4',
 'weight': 1.0,
 'inter': ['SKOV3_OVARY_amplicon4_ecDNA_1'],
 'union': ['SKOV3_OVARY_amplicon4_ecDNA_1'],
 'pval': 0.6336787070595888,
 'qval': 0.6800977350972957}

In [100]:
edges[1]

{'source': 'CLVS1',
 'target': 'SPX',
 'weight': 1.0,
 'inter': ['NCIH2171_LUNG_amplicon2_ecDNA_1'],
 'union': ['NCIH2171_LUNG_amplicon2_ecDNA_1'],
 'pval': 'N/A',
 'qval': -1}

In [109]:
chi_squared_ps = []
new_edges = edges
for edge in new_edges:
    d, p_val = chi_squared(edge)
    edge['distance'] = d
    edge['p_val'] = p_val
    chi_squared_ps.append(p_val)

/Users/michael/anaconda3/lib/python3.10/site-packages/scipy/stats/_stats_py.py:7412: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs_float - f_exp)**2 / f_exp


In [111]:
chi_squared_ps_drop = [p for p in chi_squared_ps if p != 'N/A']
len(chi_squared_ps), len(chi_squared_ps_drop)

(117622, 88606)

In [114]:
sum(chi_squared_ps_drop)/len(chi_squared_ps_drop)

2.0575891480611846e-16

In [107]:
new_edges_df = pd.DataFrame(new_edges)

In [108]:
new_edges_df

,source,target,weight,inter,union,pval,qval,distance,p_val
0,BORCS8-MEF2B,HAPLN4,1.000000,[SKOV3_OVARY_amplicon4_ecDNA_1],[SKOV3_OVARY_amplicon4_ecDNA_1],0.633679,0.680098,109189,0.0
1,CLVS1,SPX,1.000000,[NCIH2171_LUNG_amplicon2_ecDNA_1],[NCIH2171_LUNG_amplicon2_ecDNA_1],N/A,-1.000000,N/A,N/A
2,DRG2,PEMT,1.000000,[SNU182_LIVER_amplicon1_ecDNA_1],[SNU182_LIVER_amplicon1_ecDNA_1],0.335952,0.450529,582385,0.0
3,DUSP13,TMEM254,0.500000,[EFM19_BREAST_amplicon3_ecDNA_1],"[EFM19_BREAST_amplicon3_ecDNA_1, PC3_PROSTATE_...",0.011867,0.032621,4984214,0.0
4,B4GALNT2,UCKL1,0.333333,[HCC1419_BREAST_amplicon1_ecDNA_2],"[HCC1419_BREAST_amplicon1_ecDNA_2, BT474_BREAS...",N/A,-1.000000,N/A,N/A
...,...,...,...,...,...,...,...,...,...
117617,ME3,PLEKHG6,1.000000,[NCIH2171_LUNG_amplicon2_ecDNA_2],[NCIH2171_LUNG_amplicon2_ecDNA_2],N/A,-1.000000,N/A,N/A
117618,OXCT2,MYCBP,0.272727,"[NCIH889_LUNG_amplicon1_ecDNA_1, NCIH1184_LUNG...","[NCIH1836_LUNG_amplicon2_ecDNA_1, CORL88_LUNG_...",0.243684,0.364765,907033,0.0
117619,PIPOX,SNORD42A,0.500000,[KYSE410_OESOPHAGUS_amplicon3_ecDNA_1],"[OE19_OESOPHAGUS_amplicon2_ecDNA_1, KYSE410_OE...",0.456059,0.547830,319712,0.0
117620,INSM2,RBM23,0.333333,[NCIH2029_LUNG_amplicon2_ecDNA_1],"[EBC1_LUNG_amplicon19_ecDNA_1, NCIH2029_LUNG_a...",0.000093,0.000556,12640960,0.0


In [128]:
# chi square without assuming independence
chi_squared_dep_ps = []

for edge in new_edges:
    d, p_val = chi_squared_dep(edge)
    edge['distance'] = d
    edge['p_val'] = p_val
    chi_squared_dep_ps.append(p_val)

/Users/michael/anaconda3/lib/python3.10/site-packages/scipy/stats/_stats_py.py:7412: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs_float - f_exp)**2 / f_exp


In [130]:
chi_squared_dep_ps_drop = [p for p in chi_squared_dep_ps if p != -1]
len(chi_squared_dep_ps), len(chi_squared_dep_ps_drop)

(117622, 88606)

In [131]:
sum(chi_squared_dep_ps_drop)/len(chi_squared_dep_ps_drop)

0.04890102709084436

In [132]:
new_edges_df = pd.DataFrame(new_edges)

In [144]:
oncogenes = []

# Open the file and read each line into the list
with open('/Users/michael/Downloads/oncogene_list_hg38.txt', 'r') as f:
    oncogenes = [line.strip() for line in f]  

In [145]:
def oncogene_number(a, b):
    return int(a in oncogenes) + int(b in oncogenes)

In [146]:
new_edges_df['oncogene_number'] = new_edges_df.apply(
    lambda row: oncogene_number(row['source'], row['target']), axis=1
)

In [162]:
sig_edges_df = new_edges_df[(new_edges_df['p_val'] != -1) & (new_edges_df['p_val'] < 0.05) & (new_edges_df['oncogene_number'] == 0)]

In [163]:
sig_edges_df.sort_values(['p_val'])

,source,target,weight,inter,union,pval,qval,distance,p_val,oncogene_number
117620,INSM2,RBM23,0.333333,[NCIH2029_LUNG_amplicon2_ecDNA_1],"[EBC1_LUNG_amplicon19_ecDNA_1, NCIH2029_LUNG_a...",0.000093,5.558694e-04,12640960,0.000000,0
36251,ACOD1,MLNR,1.000000,[UACC812_BREAST_amplicon1_ecDNA_3],[UACC812_BREAST_amplicon1_ecDNA_3],0.0,1.221031e-07,27728173,0.000000,0
87072,CA4,MED24,0.200000,[BT474_BREAST_amplicon1_ecDNA_3],"[UACC812_BREAST_amplicon1_ecDNA_3, BT474_BREAS...",0.000001,7.385737e-06,20130869,0.000000,0
36247,CISD3,ELAC2,0.250000,[UACC812_BREAST_amplicon1_ecDNA_3],"[UACC812_BREAST_amplicon1_ecDNA_3, SNU182_LIVE...",0.0,3.477744e-07,25738729,0.000000,0
36240,EIF3H,SLCO5A1,0.500000,[EFM192A_BREAST_amplicon1_ecDNA_6],"[HCC1954_BREAST_amplicon2_ecDNA_3, EFM192A_BRE...",0.0,4.025091e-12,46975085,0.000000,0
...,...,...,...,...,...,...,...,...,...,...
94819,ZNF684,MEAF6,0.200000,[NCIH1184_LUNG_amplicon1_ecDNA_1],"[CORL88_LUNG_amplicon3_ecDNA_1, NCIH1184_LUNG_...",0.045213,1.006256e-01,3041676,0.049955,0
73509,NT5M,ZNF287,1.000000,[SNU182_LIVER_amplicon1_ecDNA_1],[SNU182_LIVER_amplicon1_ecDNA_1],0.281382,4.012895e-01,756419,0.049961,0
101577,CERS2,RIIAD1,1.000000,[KARPAS299_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE_...,[KARPAS299_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE_...,0.281401,4.012977e-01,756349,0.049968,0
8548,LYZL1,WAC,1.000000,[NCIH1693_LUNG_amplicon3_ecDNA_1],[NCIH1693_LUNG_amplicon3_ecDNA_1],0.281417,4.013076e-01,756291,0.049973,0


In [ ]:
0:51518/101697
1:8191/15331
2:320/594

In [156]:
Counter(new_edges_df['oncogene_number'])

Counter({0: 101697, 1: 15331, 2: 594})

In [ ]:
# NOTES FOR INTEGRATION INTO COAMP
# 1. Add total_samples member variable: length of dataset
# 2. For each edge, add xor properties in addition to inter and union: A and not B, B and not A